# Cache Audit Notebook
**Goal:** Figure out exactly what's in the annotation cache, what's in the TMDB cache, how they relate to your enriched film records, and what still needs annotating.

Run cells top to bottom.

In [1]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ── Paths ──────────────────────────────────────────────────────
ENRICHED_PATH      = Path('data/processed/enriched_films.json')
ANNOTATION_CACHE   = Path('data/cache/annotations')
TMDB_CACHE         = Path('data/cache/tmdb')
RATINGS_CSV        = Path('data/raw/ratings.csv')

print('Paths exist:')
for p in [ENRICHED_PATH, ANNOTATION_CACHE, TMDB_CACHE, RATINGS_CSV]:
    print(f'  {p}: {p.exists()}')

Paths exist:
  data/processed/enriched_films.json: True
  data/cache/annotations: True
  data/cache/tmdb: True
  data/raw/ratings.csv: True


## 1. What's in the enriched records file?

In [2]:
enriched_raw = json.loads(ENRICHED_PATH.read_text())
df_enriched = pd.DataFrame(enriched_raw)

print(f'Total enriched records:        {len(df_enriched)}')
print(f'Columns: {list(df_enriched.columns)}')
print()
print(f'Enriched=True:                 {df_enriched["enriched"].sum()}')
print(f'Has tmdb_id:                   {df_enriched["tmdb_id"].notna().sum()}')
print(f'Has rating:                    {df_enriched["rating"].notna().sum()}')
print(f'Has rating + enriched + tmdb:  {((df_enriched["rating"].notna()) & (df_enriched["enriched"]==True) & (df_enriched["tmdb_id"].notna())).sum()}')
if 'is_tv' in df_enriched.columns:
    print(f'Flagged as TV:                 {df_enriched["is_tv"].sum()}')

Total enriched records:        819
Columns: ['title', 'year', 'rating', 'watch_date', 'letterboxd_uri', 'review', 'tmdb_id', 'imdb_id', 'runtime', 'original_language', 'production_countries', 'genres', 'overview', 'tmdb_rating', 'tmdb_vote_count', 'director', 'cinematographer', 'editor', 'writer', 'composer', 'enriched', 'enrichment_error']

Enriched=True:                 808
Has tmdb_id:                   808
Has rating:                    819
Has rating + enriched + tmdb:  808


## 2. What's in the annotation cache?

In [3]:
annotation_files = list(ANNOTATION_CACHE.glob('*.json'))
print(f'Annotation cache files: {len(annotation_files)}')

annotations = []
bad_files = []
for f in annotation_files:
    try:
        data = json.loads(f.read_text())
        data['_cache_file_id'] = int(f.stem)  # the filename ID
        annotations.append(data)
    except Exception as e:
        bad_files.append((f.name, str(e)))

df_ann = pd.DataFrame(annotations)
print(f'Successfully parsed:    {len(df_ann)}')
print(f'Corrupt/unreadable:     {len(bad_files)}')
if bad_files:
    print(f'Bad files: {bad_files[:5]}')
print()
print('Annotation columns:', list(df_ann.columns))

Annotation cache files: 118
Successfully parsed:    118
Corrupt/unreadable:     0

Annotation columns: ['tmdb_id', 'title', 'year', 'runtime_minutes', 'original_language', 'narrative_time', 'pacing_signature', 'point_of_view', 'ending_valence', 'tone_primary', 'tone_secondary', 'reality_register', 'moral_complexity', 'character_legibility', 'dialogue_density', 'cinematographic_language', 'colour_palette', 'aspect_ratio', 'score_style', 'editor_signature', 'world_building_depth', 'genre_subversion', 'thematic_primary', 'thematic_secondary', 'director_lineage', 'body_experience', 'production_register', 'user_rating', 'user_review_text', 'watch_date', 'letterboxd_avg_rating', 'metacritic_score', 'critical_divergence', 'annotation_model', 'annotation_version', 'annotation_confidence', 'confidence_note', '_cache_file_id']


## 3. Do annotation file IDs match their internal tmdb_id field?

In [4]:
# The filename stem should equal the tmdb_id field inside the JSON
if 'tmdb_id' in df_ann.columns:
    df_ann['tmdb_id'] = pd.to_numeric(df_ann['tmdb_id'], errors='coerce')
    df_ann['id_match'] = df_ann['tmdb_id'] == df_ann['_cache_file_id']
    
    mismatches = df_ann[~df_ann['id_match']]
    print(f'Files where filename ID == internal tmdb_id: {df_ann["id_match"].sum()}')
    print(f'MISMATCHES (filename != internal tmdb_id):   {len(mismatches)}')
    
    if len(mismatches) > 0:
        print('\nSample mismatches:')
        print(mismatches[['_cache_file_id', 'tmdb_id', 'title']].head(10).to_string())
else:
    print('No tmdb_id field found in annotations — schema mismatch')
    print('Fields present:', list(df_ann.columns))

Files where filename ID == internal tmdb_id: 118
MISMATCHES (filename != internal tmdb_id):   0


## 4. Cross-reference: which annotation cache files are in your enriched records?

In [5]:
enriched_tmdb_ids = set(
    int(r['tmdb_id']) for r in enriched_raw 
    if r.get('tmdb_id') and r.get('enriched')
)
cache_file_ids = set(int(f.stem) for f in annotation_files)

# Use internal tmdb_id from annotation JSON, not just filename
cache_internal_ids = set(
    int(a['tmdb_id']) for a in annotations 
    if a.get('tmdb_id')
) if 'tmdb_id' in df_ann.columns else set()

in_both_by_filename   = enriched_tmdb_ids & cache_file_ids
in_both_by_internal   = enriched_tmdb_ids & cache_internal_ids
rogue_by_filename     = cache_file_ids - enriched_tmdb_ids
missing_from_cache    = enriched_tmdb_ids - cache_file_ids

print(f'Enriched films (with tmdb_id):          {len(enriched_tmdb_ids)}')
print(f'Cache files (by filename):              {len(cache_file_ids)}')
print(f'Cache files (by internal tmdb_id):      {len(cache_internal_ids)}')
print()
print(f'Matched by filename:                    {len(in_both_by_filename)}')
print(f'Matched by internal tmdb_id:            {len(in_both_by_internal)}')
print(f'Cache files NOT in enriched (rogue):    {len(rogue_by_filename)}')
print(f'Enriched films NOT in cache (missing):  {len(missing_from_cache)}')

Enriched films (with tmdb_id):          808
Cache files (by filename):              118
Cache files (by internal tmdb_id):      118

Matched by filename:                    118
Matched by internal tmdb_id:            118
Cache files NOT in enriched (rogue):    0
Enriched films NOT in cache (missing):  690


## 5. Inspect the rogue cache files — what are they?

In [6]:
rogue_annotations = df_ann[df_ann['_cache_file_id'].isin(rogue_by_filename)].copy()
print(f'Rogue annotation files: {len(rogue_annotations)}')
print()

if 'title' in rogue_annotations.columns:
    print('Sample rogue files (title from JSON):')
    sample_cols = ['_cache_file_id', 'tmdb_id', 'title']
    available = [c for c in sample_cols if c in rogue_annotations.columns]
    print(rogue_annotations[available].head(20).to_string())
else:
    print('No title field — showing raw keys:')
    print(rogue_annotations.head(5).to_string())

Rogue annotation files: 957

Sample rogue files (title from JSON):
    _cache_file_id  tmdb_id                           title
0              683      683                      The Player
2           921703   921703                     One Day Off
3            76298    76298                             Her
4             4908     4908                     Le Samouraï
5            69478    69478                 Incomplete Life
6              396      396              This Is Spinal Tap
7           126895   126895                      Talk to Me
8           108247   108247                       Shithouse
10          377492   377492                   The Favourite
11          946147   946147                        Vellakka
12          748291   748291                         Annette
13          726861   726861                           Cargo
14            6431     6431                   Fallen Angels
15         1390486  1390486                  Operation Java
16             142      142      

## 6. Check the TMDB search cache — how does it map titles to IDs?

In [7]:
tmdb_files = list(TMDB_CACHE.glob('*.json'))
search_files  = [f for f in tmdb_files if f.stem.startswith('search_')]
details_files = [f for f in tmdb_files if f.stem.startswith('details_')]
credits_files = [f for f in tmdb_files if f.stem.startswith('credits_')]

print(f'TMDB cache files total:   {len(tmdb_files)}')
print(f'  search_*:               {len(search_files)}')
print(f'  details_*:              {len(details_files)}')
print(f'  credits_*:              {len(credits_files)}')
print()

# Parse search cache — each contains {tmdb_id: X}
search_map = {}  # title_year_key -> tmdb_id
for f in search_files:
    try:
        data = json.loads(f.read_text())
        key = f.stem.replace('search_', '')
        search_map[key] = data.get('tmdb_id')
    except:
        pass

search_ids = set(v for v in search_map.values() if v)
details_ids = set(int(f.stem.replace('details_', '')) for f in details_files)

print(f'Unique TMDB IDs in search cache:  {len(search_ids)}')
print(f'Unique TMDB IDs in details cache: {len(details_ids)}')
print()

# Are annotation cache IDs coming from the details cache?
ann_ids_in_details  = cache_file_ids & details_ids
ann_ids_NOT_details = cache_file_ids - details_ids
print(f'Annotation IDs also in details cache:     {len(ann_ids_in_details)}')
print(f'Annotation IDs NOT in details cache:      {len(ann_ids_NOT_details)}')
if ann_ids_NOT_details:
    print(f'Sample annotation IDs not in details: {list(ann_ids_NOT_details)[:10]}')

TMDB cache files total:   2434
  search_*:               818
  details_*:              808
  credits_*:              808

Unique TMDB IDs in search cache:  808
Unique TMDB IDs in details cache: 808

Annotation IDs also in details cache:     38
Annotation IDs NOT in details cache:      957
Sample annotation IDs not in details: [1, 2, 7, 1392648, 487433, 319498, 9, 12, 14, 16398]


## 7. For rated films — what's the annotation status?

In [8]:
rated_enriched = [
    r for r in enriched_raw
    if r.get('rating') is not None
    and r.get('enriched') == True
    and r.get('tmdb_id') is not None
    and not r.get('is_tv', False)
]

print(f'Rated + enriched + has tmdb_id: {len(rated_enriched)}')

has_annotation = []
no_annotation  = []

for r in rated_enriched:
    tid = int(r['tmdb_id'])
    if tid in cache_file_ids:
        has_annotation.append(r)
    else:
        no_annotation.append(r)

print(f'Has annotation (by filename):   {len(has_annotation)}')
print(f'Missing annotation:             {len(no_annotation)}')
print()

if no_annotation:
    print('Sample films missing annotation:')
    for r in no_annotation[:15]:
        print(f"  [{r['rating']}★] {r['title']} ({r['year']}) — tmdb_id={r['tmdb_id']}")

Rated + enriched + has tmdb_id: 808
Has annotation (by filename):   38
Missing annotation:             770

Sample films missing annotation:
  [4.5★] Bo Burnham: Inside (2021) — tmdb_id=823754
  [4.5★] Ship of Theseus (2012) — tmdb_id=128207
  [4.0★] Inside Llewyn Davis (2013) — tmdb_id=86829
  [3.0★] Fear Street: 1978 (2021) — tmdb_id=591274
  [4.0★] Malik (2021) — tmdb_id=665133
  [4.5★] Chungking Express (1994) — tmdb_id=11104
  [4.0★] First Cow (2019) — tmdb_id=558582
  [4.5★] Shoplifters (2018) — tmdb_id=505192
  [4.5★] A Single Man (2009) — tmdb_id=34653
  [1.0★] Shiva Baby (2020) — tmdb_id=664300
  [4.0★] The Suicide Squad (2021) — tmdb_id=436969
  [4.5★] Mommy (2014) — tmdb_id=265177
  [4.0★] The Kid Detective (2020) — tmdb_id=720755
  [3.0★] Fear Street: 1994 (2021) — tmdb_id=591273
  [2.0★] Annette (2021) — tmdb_id=424277


## 8. Sample the actual annotation content — does the schema look right?

In [9]:
# Cross-reference: find annotations that belong to rated films
rated_ids = set(int(r['tmdb_id']) for r in rated_enriched)
rated_annotations = df_ann[df_ann['_cache_file_id'].isin(rated_ids)]

print(f'Annotations belonging to rated films: {len(rated_annotations)}')
print()

# Check which schema fields are present
new_schema_fields = [
    'tone_primary', 'thematic_primary', 'reality_register',
    'body_experience', 'production_register', 'ending_valence',
    'moral_complexity', 'point_of_view', 'character_legibility'
]
old_schema_fields = ['tone', 'thematic_cluster', 'moral_complexity']

print('Schema version check:')
for field in new_schema_fields:
    present = field in df_ann.columns
    print(f'  {field}: {"✓" if present else "✗ MISSING"}')

print()
# Show a clean sample of one annotation
if len(rated_annotations) > 0:
    sample = rated_annotations.iloc[0].dropna().to_dict()
    print('Sample annotation (one rated film):')
    for k, v in sample.items():
        if not k.startswith('_'):
            print(f'  {k}: {v}')

Annotations belonging to rated films: 38

Schema version check:
  tone_primary: ✓
  thematic_primary: ✓
  reality_register: ✓
  body_experience: ✓
  production_register: ✓
  ending_valence: ✓
  moral_complexity: ✓
  point_of_view: ✓
  character_legibility: ✓

Sample annotation (one rated film):
  tmdb_id: 379
  title: 1917
  narrative_time: non_linear
  pacing_signature: propulsive
  point_of_view: omniscient_detached
  ending_valence: cathartic_resolved
  tone_primary: tense
  tone_secondary: []
  reality_register: hyperrealist
  moral_complexity: sympathetic_antagonist
  character_legibility: behaviorist_opaque
  dialogue_density: sparse
  cinematographic_language: static_composed
  colour_palette: desaturated_muted
  aspect_ratio: widescreen_185
  score_style: ambient
  editor_signature: rhythmic_expressive
  world_building_depth: immersive
  genre_subversion: none
  thematic_primary: memory_time
  thematic_secondary: []
  director_lineage: mumblecore_naturalism
  body_experience: p

## 9. Summary + recommended action

In [10]:
print('=' * 55)
print('AUDIT SUMMARY')
print('=' * 55)
print(f'Enriched records total:              {len(enriched_raw)}')
print(f'Rated + enriched + has tmdb_id:      {len(rated_enriched)}')
print(f'Annotation cache files:              {len(annotation_files)}')
print(f'Rated films WITH annotation:         {len(has_annotation)}')
print(f'Rated films MISSING annotation:      {len(no_annotation)}')
print(f'Rogue cache files (not in enriched): {len(rogue_by_filename)}')
print()

if len(no_annotation) == 0:
    print('✅ All rated films are annotated. Ready for Step 4.')
elif len(no_annotation) < 50:
    print(f'⚠️  {len(no_annotation)} rated films need annotation.')
    print('   Recommendation: targeted re-run on missing films only.')
else:
    print(f'🔴 {len(no_annotation)} rated films still need annotation.')
    print('   Recommendation: clean rogue files and restart annotation run.')

print()
if len(rogue_by_filename) > 100:
    print(f'🔴 {len(rogue_by_filename)} rogue cache files found.')
    print('   These are wasting disk space but not causing errors.')
    print('   Safe to delete: they are not from your rated film history.')

# Write missing tmdb_ids to a file for targeted re-run
if no_annotation:
    missing_ids = [int(r['tmdb_id']) for r in no_annotation]
    Path('data/processed/missing_annotation_ids.json').write_text(
        json.dumps(missing_ids, indent=2)
    )
    print(f'\nWrote {len(missing_ids)} missing tmdb_ids to:')
    print('  data/processed/missing_annotation_ids.json')
    print('  (Use this for targeted re-run)')

AUDIT SUMMARY
Enriched records total:              819
Rated + enriched + has tmdb_id:      808
Annotation cache files:              995
Rated films WITH annotation:         38
Rated films MISSING annotation:      770
Rogue cache files (not in enriched): 957

🔴 770 rated films still need annotation.
   Recommendation: clean rogue files and restart annotation run.

🔴 957 rogue cache files found.
   These are wasting disk space but not causing errors.
   Safe to delete: they are not from your rated film history.

Wrote 770 missing tmdb_ids to:
  data/processed/missing_annotation_ids.json
  (Use this for targeted re-run)


In [21]:
# Cell 10 — Fuzzy remap rogue annotation files to canonical enriched IDs

from pathlib import Path
import json
from difflib import SequenceMatcher

# Load canonical enriched records
enriched = json.loads(Path('data/processed/enriched_films.json').read_text())

# Build lookup: (title_normalized, year) -> canonical tmdb_id
def normalize(title: str) -> str:
    import re
    title = title.lower().strip()
    title = re.sub(r'[^\w\s]', '', title)  # remove punctuation
    title = re.sub(r'\s+', ' ', title)
    return title

canonical_map = {}  # normalized_title -> {year, tmdb_id, title}
for r in enriched:
    if not r.get('tmdb_id') or not r.get('enriched'):
        continue
    key = normalize(r['title'])
    canonical_map[key] = {
        'tmdb_id': int(r['tmdb_id']),
        'title': r['title'],
        'year': r.get('year'),
    }

# Load rogue annotation files
cache = Path('data/cache/annotations')
valid_ids = set(int(r['tmdb_id']) for r in enriched if r.get('tmdb_id') and r.get('enriched'))
rogue_files = [f for f in cache.glob('*.json') if int(f.stem) not in valid_ids]

print(f"Rogue files to attempt remapping: {len(rogue_files)}")
print(f"Canonical enriched films:         {len(canonical_map)}")

# Fuzzy match each rogue file to a canonical record
def fuzzy_score(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

results = {
    'remapped':     [],  # confident match found
    'ambiguous':    [],  # multiple close matches
    'no_match':     [],  # nothing close enough
    'already_have': [],  # canonical ID already has an annotation
}

CONFIDENCE_THRESHOLD = 0.85  # tune this if needed

for f in rogue_files:
    try:
        data = json.loads(f.read_text())
    except Exception:
        results['no_match'].append({'file': f.name, 'reason': 'corrupt'})
        continue

    ann_title = data.get('title', '')
    ann_year  = data.get('year')
    if not ann_title:
        results['no_match'].append({'file': f.name, 'reason': 'no title in annotation'})
        continue

    norm_ann_title = normalize(ann_title)

    # Score against all canonical titles
    scored = []
    for canon_key, canon in canonical_map.items():
        score = fuzzy_score(norm_ann_title, canon_key)
        # Boost score if year matches
        if ann_year and canon.get('year') and int(ann_year) == int(canon['year']):
            score = min(score + 0.05, 1.0)
        scored.append((score, canon))

    scored.sort(key=lambda x: x[0], reverse=True)
    best_score, best_match = scored[0]
    second_score = scored[1][0] if len(scored) > 1 else 0

    canonical_id = best_match['tmdb_id']
    canonical_ann_path = cache / f"{canonical_id}.json"

    if best_score < CONFIDENCE_THRESHOLD:
        results['no_match'].append({
            'file': f.name,
            'ann_title': ann_title,
            'best_match': best_match['title'],
            'score': round(best_score, 3),
        })
    elif canonical_ann_path.exists():
        results['already_have'].append({
            'file': f.name,
            'ann_title': ann_title,
            'canonical': best_match['title'],
            'canonical_id': canonical_id,
            'score': round(best_score, 3),
        })
    elif best_score - second_score < 0.05 and best_score < 0.95:
        results['ambiguous'].append({
            'file': f.name,
            'ann_title': ann_title,
            'match_1': f"{scored[0][1]['title']} ({round(scored[0][0],3)})",
            'match_2': f"{scored[1][1]['title']} ({round(scored[1][0],3)})",
        })
    else:
        results['remapped'].append({
            'rogue_file': f,
            'ann_title': ann_title,
            'canonical_title': best_match['title'],
            'canonical_id': canonical_id,
            'score': round(best_score, 3),
            'data': data,
        })

print(f"\nResults:")
print(f"  ✅ Confident remap:      {len(results['remapped'])}")
print(f"  ✅ Already have:         {len(results['already_have'])} (rogue is a duplicate)")
print(f"  ⚠️  Ambiguous match:      {len(results['ambiguous'])}")
print(f"  ❌ No match found:       {len(results['no_match'])}")

Rogue files to attempt remapping: 0
Canonical enriched films:         806

Results:
  ✅ Confident remap:      0
  ✅ Already have:         0 (rogue is a duplicate)
  ⚠️  Ambiguous match:      0
  ❌ No match found:       0


In [22]:
# Cell 11 — Preview remaps before writing anything

print("Sample confident remaps (first 20):")
print(f"{'Rogue file':<12} {'Annotation title':<35} {'→ Canonical title':<35} {'Score'}")
print("-" * 90)
for r in results['remapped'][:20]:
    print(f"{r['rogue_file'].name:<12} {r['ann_title'][:33]:<35} {r['canonical_title'][:33]:<35} {r['score']}")

print(f"\nSample ambiguous (need manual review):")
for r in results['ambiguous'][:10]:
    print(f"  {r['ann_title']}: '{r['match_1']}' vs '{r['match_2']}'")

print(f"\nSample no-match:")
for r in results['no_match'][:10]:
    print(f"  {r.get('ann_title', '?')} → best: {r.get('best_match', 'none')} ({r.get('score', '?')})")

Sample confident remaps (first 20):
Rogue file   Annotation title                    → Canonical title                   Score
------------------------------------------------------------------------------------------

Sample ambiguous (need manual review):

Sample no-match:


In [13]:
# Cell 14 — Find all duplicate canonical IDs (multiple rogue files mapping to same enriched film)

from collections import defaultdict
import shutil

DUPLICATES_DIR = Path('data/cache/annotations_duplicates')
DUPLICATES_DIR.mkdir(parents=True, exist_ok=True)

# Build a map: canonical_tmdb_id -> list of all files that claim to be that film
# This includes both the canonical file (if exists) and any remapped rogues

canonical_to_files = defaultdict(list)

# First, all current valid annotation files
for f in cache.glob('*.json'):
    try:
        data = json.loads(f.read_text())
        file_id = int(f.stem)
        
        # Check internal tmdb_id too
        internal_id = data.get('tmdb_id')
        title = data.get('title', '')
        
        canonical_to_files[file_id].append({
            'path': f,
            'internal_id': internal_id,
            'title': title,
            'source': 'canonical_filename',
        })
        
        # If internal ID differs from filename, it's a mismatch
        if internal_id and int(internal_id) != file_id:
            canonical_to_files[int(internal_id)].append({
                'path': f,
                'internal_id': internal_id,
                'title': title,
                'source': 'internal_id_differs',
            })
    except Exception as e:
        print(f"Could not parse {f.name}: {e}")

# Now add the rogue remaps from Cell 10
remap_candidates = defaultdict(list)  # canonical_id -> list of rogue files that match it
for r in results['remapped']:
    remap_candidates[r['canonical_id']].append(r)

# Find canonical IDs that have MORE than one file competing for them
duplicates = {}
for canon_id, candidates in remap_candidates.items():
    canonical_path = cache / f"{canon_id}.json"
    if canonical_path.exists() or len(candidates) > 1:
        duplicates[canon_id] = {
            'canonical_path': canonical_path,
            'has_canonical': canonical_path.exists(),
            'rogue_candidates': candidates,
        }

print(f"Canonical IDs with duplicate candidates: {len(duplicates)}")
print(f"Duplicates output folder: {DUPLICATES_DIR}")
print()

# Sample
for canon_id, info in list(duplicates.items())[:5]:
    print(f"\nTMDB ID {canon_id}:")
    print(f"  Canonical file exists: {info['has_canonical']}")
    for c in info['rogue_candidates']:
        print(f"  Rogue: {c['rogue_file'].name} → '{c['ann_title']}' (score={c['score']})")

Canonical IDs with duplicate candidates: 406
Duplicates output folder: data/cache/annotations_duplicates


TMDB ID 10403:
  Canonical file exists: False
  Rogue: 683.json → 'The Player' (score=1.0)
  Rogue: 6821.json → 'The Player' (score=1.0)

TMDB ID 1326106:
  Canonical file exists: False
  Rogue: 921703.json → 'One Day Off' (score=1.0)
  Rogue: 1049723.json → 'One Day Off' (score=1.0)

TMDB ID 152601:
  Canonical file exists: False
  Rogue: 76298.json → 'Her' (score=1.0)
  Rogue: 397806.json → 'Her' (score=1.0)

TMDB ID 5511:
  Canonical file exists: False
  Rogue: 4908.json → 'Le Samouraï' (score=1.0)
  Rogue: 16874.json → 'Le Samouraï' (score=1.0)

TMDB ID 541176:
  Canonical file exists: False
  Rogue: 69478.json → 'Incomplete Life' (score=1.0)
  Rogue: 474197.json → 'Incomplete Life' (score=1.0)


In [14]:
# Cell 15 — For each duplicate set: keep best in main cache, move others to duplicates folder

WRITE = False  # ← set True when ready to execute

kept   = []
moved  = []
skipped = []

def pick_best(candidates: list) -> dict:
    """Pick the annotation to keep — prefer higher confidence, then higher score."""
    def sort_key(c):
        data = c['data']
        conf_order = {'HIGH': 3, 'high': 3, 3: 3, 'MEDIUM': 2, 'medium': 2, 2: 2, 'LOW': 1, 'low': 1, 1: 1}
        conf = conf_order.get(data.get('annotation_confidence', 1), 1)
        return (conf, c.get('score', 0))
    return sorted(candidates, key=sort_key, reverse=True)[0]

for canon_id, info in duplicates.items():
    candidates = info['rogue_candidates']
    canonical_path = info['canonical_path']
    
    if info['has_canonical'] and len(candidates) == 0:
        # Only the canonical file exists, no rogue competition — skip
        skipped.append(canon_id)
        continue
    
    if info['has_canonical']:
        # Canonical file already exists — move all rogues to duplicates
        for c in candidates:
            dest = DUPLICATES_DIR / f"{canon_id}_rogue_{c['rogue_file'].stem}.json"
            if WRITE:
                shutil.copy2(c['rogue_file'], dest)
                c['rogue_file'].unlink()
            moved.append({
                'canon_id': canon_id,
                'moved_to': dest.name,
                'title': c['ann_title'],
                'score': c['score'],
            })
    else:
        # No canonical file yet — pick the best rogue, move the rest
        best = pick_best(candidates)
        rest = [c for c in candidates if c['rogue_file'] != best['rogue_file']]
        
        # Write best as canonical
        best_data = best['data']
        best_data['tmdb_id'] = canon_id
        best_data['title']   = best['canonical_title']
        
        if WRITE:
            canonical_path.write_text(json.dumps(best_data, indent=2))
            best['rogue_file'].unlink()
        
        kept.append({
            'canon_id': canon_id,
            'kept_title': best['ann_title'],
            'kept_score': best['score'],
            'confidence': best['data'].get('annotation_confidence'),
        })
        
        # Move losers to duplicates folder
        for c in rest:
            dest = DUPLICATES_DIR / f"{canon_id}_alt_{c['rogue_file'].stem}.json"
            if WRITE:
                shutil.copy2(c['rogue_file'], dest)
                c['rogue_file'].unlink()
            moved.append({
                'canon_id': canon_id,
                'moved_to': dest.name,
                'title': c['ann_title'],
                'score': c['score'],
            })

print(f"{'DRY RUN' if not WRITE else 'EXECUTED'}")
print(f"  Kept as canonical:        {len(kept)}")
print(f"  Moved to duplicates/:     {len(moved)}")
print(f"  Skipped (no conflict):    {len(skipped)}")
print()

if kept:
    print("Sample kept (best candidate chosen):")
    for r in kept[:10]:
        print(f"  [{r['confidence']}] {r['kept_title']} → {r['canon_id']} (score={r['kept_score']})")

if moved:
    print("\nSample moved to duplicates/:")
    for r in moved[:10]:
        print(f"  {r['moved_to']} — '{r['title']}'")

DRY RUN
  Kept as canonical:        406
  Moved to duplicates/:     411
  Skipped (no conflict):    0

Sample kept (best candidate chosen):
  [2] The Player → 10403 (score=1.0)
  [3] One Day Off → 1326106 (score=1.0)
  [2] Her → 152601 (score=1.0)
  [3] Le Samouraï → 5511 (score=1.0)
  [3] Incomplete Life → 541176 (score=1.0)
  [3] Talk to Me → 1008042 (score=1.0)
  [3] Shithouse → 637053 (score=1.0)
  [3] The Favourite → 375262 (score=1.0)
  [2] Cargo → 640421 (score=1.0)
  [3] Fallen Angels → 11220 (score=1.0)

Sample moved to duplicates/:
  10403_alt_6821.json — 'The Player'
  1326106_alt_921703.json — 'One Day Off'
  152601_alt_397806.json — 'Her'
  5511_alt_16874.json — 'Le Samouraï'
  541176_alt_474197.json — 'Incomplete Life'
  1008042_alt_126895.json — 'Talk to Me'
  637053_alt_763892.json — 'Shithouse'
  375262_alt_468271.json — 'The Favourite'
  640421_alt_674128.json — 'Cargo'
  11220_alt_6431.json — 'Fallen Angels'


In [16]:
# Cell 15 — For each duplicate set: keep best in main cache, move others to duplicates folder

WRITE = True  # ← set True when ready to execute

kept   = []
moved  = []
skipped = []

def pick_best(candidates: list) -> dict:
    """Pick the annotation to keep — prefer higher confidence, then higher score."""
    def sort_key(c):
        data = c['data']
        conf_order = {'HIGH': 3, 'high': 3, 3: 3, 'MEDIUM': 2, 'medium': 2, 2: 2, 'LOW': 1, 'low': 1, 1: 1}
        conf = conf_order.get(data.get('annotation_confidence', 1), 1)
        return (conf, c.get('score', 0))
    return sorted(candidates, key=sort_key, reverse=True)[0]

for canon_id, info in duplicates.items():
    candidates = info['rogue_candidates']
    canonical_path = info['canonical_path']
    
    if info['has_canonical'] and len(candidates) == 0:
        # Only the canonical file exists, no rogue competition — skip
        skipped.append(canon_id)
        continue
    
    if info['has_canonical']:
        # Canonical file already exists — move all rogues to duplicates
        for c in candidates:
            dest = DUPLICATES_DIR / f"{canon_id}_rogue_{c['rogue_file'].stem}.json"
            if WRITE:
                shutil.copy2(c['rogue_file'], dest)
                c['rogue_file'].unlink()
            moved.append({
                'canon_id': canon_id,
                'moved_to': dest.name,
                'title': c['ann_title'],
                'score': c['score'],
            })
    else:
        # No canonical file yet — pick the best rogue, move the rest
        best = pick_best(candidates)
        rest = [c for c in candidates if c['rogue_file'] != best['rogue_file']]
        
        # Write best as canonical
        best_data = best['data']
        best_data['tmdb_id'] = canon_id
        best_data['title']   = best['canonical_title']
        
        if WRITE:
            canonical_path.write_text(json.dumps(best_data, indent=2))
            best['rogue_file'].unlink()
        
        kept.append({
            'canon_id': canon_id,
            'kept_title': best['ann_title'],
            'kept_score': best['score'],
            'confidence': best['data'].get('annotation_confidence'),
        })
        
        # Move losers to duplicates folder
        for c in rest:
            dest = DUPLICATES_DIR / f"{canon_id}_alt_{c['rogue_file'].stem}.json"
            if WRITE:
                shutil.copy2(c['rogue_file'], dest)
                c['rogue_file'].unlink()
            moved.append({
                'canon_id': canon_id,
                'moved_to': dest.name,
                'title': c['ann_title'],
                'score': c['score'],
            })

print(f"{'DRY RUN' if not WRITE else 'EXECUTED'}")
print(f"  Kept as canonical:        {len(kept)}")
print(f"  Moved to duplicates/:     {len(moved)}")
print(f"  Skipped (no conflict):    {len(skipped)}")
print()

if kept:
    print("Sample kept (best candidate chosen):")
    for r in kept[:10]:
        print(f"  [{r['confidence']}] {r['kept_title']} → {r['canon_id']} (score={r['kept_score']})")

if moved:
    print("\nSample moved to duplicates/:")
    for r in moved[:10]:
        print(f"  {r['moved_to']} — '{r['title']}'")

EXECUTED
  Kept as canonical:        406
  Moved to duplicates/:     411
  Skipped (no conflict):    0

Sample kept (best candidate chosen):
  [2] The Player → 10403 (score=1.0)
  [3] One Day Off → 1326106 (score=1.0)
  [2] Her → 152601 (score=1.0)
  [3] Le Samouraï → 5511 (score=1.0)
  [3] Incomplete Life → 541176 (score=1.0)
  [3] Talk to Me → 1008042 (score=1.0)
  [3] Shithouse → 637053 (score=1.0)
  [3] The Favourite → 375262 (score=1.0)
  [2] Cargo → 640421 (score=1.0)
  [3] Fallen Angels → 11220 (score=1.0)

Sample moved to duplicates/:
  10403_alt_6821.json — 'The Player'
  1326106_alt_921703.json — 'One Day Off'
  152601_alt_397806.json — 'Her'
  5511_alt_16874.json — 'Le Samouraï'
  541176_alt_474197.json — 'Incomplete Life'
  1008042_alt_126895.json — 'Talk to Me'
  637053_alt_763892.json — 'Shithouse'
  375262_alt_468271.json — 'The Favourite'
  640421_alt_674128.json — 'Cargo'
  11220_alt_6431.json — 'Fallen Angels'


In [15]:
# Cell 16 — Compare a duplicate pair side by side so you can review manually

import pandas as pd

def compare_annotations(canon_id: int):
    """Show a side-by-side diff of all annotation files for a given tmdb_id."""
    main_file = cache / f"{canon_id}.json"
    dup_files = list(DUPLICATES_DIR.glob(f"{canon_id}_*.json"))
    
    all_files = ([main_file] if main_file.exists() else []) + dup_files
    if not all_files:
        print(f"No files found for tmdb_id {canon_id}")
        return
    
    records = {}
    for f in all_files:
        try:
            data = json.loads(f.read_text())
            records[f.name] = data
        except Exception as e:
            print(f"Could not read {f.name}: {e}")
    
    # Build comparison dataframe
    craft_fields = [
        'title', 'narrative_time', 'pacing_signature', 'point_of_view',
        'ending_valence', 'tone_primary', 'tone_secondary', 'reality_register',
        'moral_complexity', 'character_legibility', 'dialogue_density',
        'cinematographic_language', 'colour_palette', 'aspect_ratio',
        'score_style', 'editor_signature', 'world_building_depth',
        'genre_subversion', 'thematic_primary', 'thematic_secondary',
        'director_lineage', 'body_experience', 'production_register',
        'annotation_confidence',
    ]
    
    comparison = {}
    for field in craft_fields:
        row = {}
        for fname, data in records.items():
            row[fname] = data.get(field, '—')
        # Flag if values differ
        unique_vals = set(str(v) for v in row.values())
        row['_differs'] = '⚠️' if len(unique_vals) > 1 else '✓'
        comparison[field] = row
    
    df = pd.DataFrame(comparison).T
    
    # Show only rows where values differ
    differs = df[df['_differs'] == '⚠️'].drop(columns='_differs')
    same    = df[df['_differs'] == '✓'].drop(columns='_differs')
    
    print(f"\n{'='*60}")
    print(f"TMDB ID {canon_id} — {list(records.values())[0].get('title', '?')}")
    print(f"{'='*60}")
    print(f"\n⚠️  DIFFERING FIELDS ({len(differs)}):")
    if len(differs):
        print(differs.to_string())
    else:
        print("  None — all annotations agree!")
    print(f"\n✓  MATCHING FIELDS ({len(same)}): {list(same.index)}")

# Example usage — replace with any canon_id from your duplicates
if duplicates:
    sample_id = list(duplicates.keys())[0]
    compare_annotations(sample_id)

No files found for tmdb_id 10403


In [ ]:
# Cell 12 — Write remapped files + delete rogues
# ⚠️  Only run this after reviewing Cell 11 output and confirming remaps look correct

WRITE = True  # set to False for a dry run first

remapped_count  = 0
deleted_count   = 0
skipped_count   = 0

for r in results['remapped']:
    canonical_id   = r['canonical_id']
    rogue_file     = r['rogue_file']
    data           = r['data']
    canonical_path = cache / f"{canonical_id}.json"

    if canonical_path.exists():
        # Canonical already has a file — rogue is a duplicate, just delete
        if WRITE:
            rogue_file.unlink()
        deleted_count += 1
        continue

    # Write annotation with corrected tmdb_id
    data['tmdb_id'] = canonical_id
    data['title']   = r['canonical_title']

    if WRITE:
        canonical_path.write_text(json.dumps(data, indent=2))
        rogue_file.unlink()
        remapped_count += 1
    else:
        print(f"DRY RUN: would write {canonical_path.name} for '{r['ann_title']}'")
        remapped_count += 1

# Delete files we already have canonical annotations for
for r in results['already_have']:
    rogue_path = cache / r['file']
    if WRITE and rogue_path.exists():
        rogue_path.unlink()
        deleted_count += 1

print(f"\n{'DRY RUN' if not WRITE else 'WRITE'} complete:")
print(f"  Remapped (new canonical file written): {remapped_count}")
print(f"  Deleted (duplicate of existing):       {deleted_count}")
print(f"  Skipped (ambiguous or no match):       {skipped_count}")
print(f"  Still need annotation:                 {len(results['ambiguous']) + len(results['no_match'])}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/cache/annotations/683.json'

In [24]:
# Cell 13 — Final state after remap

cache_files = list(cache.glob('*.json'))
valid_after = [f for f in cache_files if int(f.stem) in valid_ids]
still_rogue = [f for f in cache_files if int(f.stem) not in valid_ids]

rated_enriched_ids = set(
    int(r['tmdb_id']) for r in enriched
    if r.get('rating') and r.get('enriched') and r.get('tmdb_id')
)
now_annotated = rated_enriched_ids & set(int(f.stem) for f in valid_after)
still_missing = rated_enriched_ids - set(int(f.stem) for f in valid_after)

print("=" * 50)
print("FINAL STATE AFTER REMAP")
print("=" * 50)
print(f"Total cache files:              {len(cache_files)}")
print(f"Valid (in enriched):            {len(valid_after)}")
print(f"Still rogue:                    {len(still_rogue)}")
print(f"Rated films now annotated:      {len(now_annotated)}")
print(f"Rated films still missing:      {len(still_missing)}")

# Update the missing IDs file
if still_missing:
    Path('data/processed/missing_annotation_ids.json').write_text(
        json.dumps(list(still_missing), indent=2)
    )
    print(f"\nUpdated missing_annotation_ids.json with {len(still_missing)} films")
    print("Sample still missing:")
    id_to_title = {int(r['tmdb_id']): r['title'] for r in enriched if r.get('tmdb_id')}
    for mid in list(still_missing)[:10]:
        print(f"  {mid}: {id_to_title.get(mid, 'unknown')}")

FINAL STATE AFTER REMAP
Total cache files:              444
Valid (in enriched):            444
Still rogue:                    0
Rated films now annotated:      444
Rated films still missing:      364

Updated missing_annotation_ids.json with 364 films
Sample still missing:
  1114114: IB 71
  10242: What Ever Happened to Baby Jane?
  1118224: Maharaja
  1177623: Rifle Club
  313369: La La Land
  10276: What About Bob?
  647206: Android Kunjappan Version 5.25
  22584: To Have and Have Not
  59: A History of Violence
  1151039: Nonnas


In [23]:
# Cell 18 — Why are well-known films still missing?

still_missing_ids = json.loads(Path('data/processed/missing_annotation_ids.json').read_text())
id_to_record = {int(r['tmdb_id']): r for r in enriched_raw if r.get('tmdb_id')}

# Check if they exist in the duplicates folder
dup_dir = Path('data/cache/annotations_duplicates')
dup_files = {int(f.stem.split('_')[0]) for f in dup_dir.glob('*.json')}

print("Missing films that ARE in duplicates folder:")
found_in_dups = []
for mid in still_missing_ids:
    if mid in dup_files:
        r = id_to_record.get(mid, {})
        found_in_dups.append((mid, r.get('title', '?')))
        
for tid, title in found_in_dups[:20]:
    print(f"  {tid}: {title}")

print(f"\nTotal missing but in duplicates: {len(found_in_dups)}")
print(f"Total missing not in duplicates: {len(still_missing_ids) - len(found_in_dups)}")

Missing films that ARE in duplicates folder:

Total missing but in duplicates: 0
Total missing not in duplicates: 364


In [25]:
# Cell 19 — Compare ALL duplicate pairs and show which is better

dup_dir = Path('data/cache/annotations_duplicates')
cache   = Path('data/cache/annotations')

craft_fields = [
    'narrative_time', 'pacing_signature', 'point_of_view', 'ending_valence',
    'tone_primary', 'reality_register', 'moral_complexity', 'character_legibility',
    'dialogue_density', 'cinematographic_language', 'colour_palette', 'aspect_ratio',
    'score_style', 'editor_signature', 'world_building_depth', 'genre_subversion',
    'thematic_primary', 'director_lineage', 'body_experience', 'production_register',
    'annotation_confidence',
]

conf_rank = {'HIGH': 3, 'high': 3, 3: 3, 'MEDIUM': 2, 'medium': 2, 2: 2, 'LOW': 1, 'low': 1, 1: 1}

rows = []
for dup_file in dup_dir.glob('*.json'):
    # Filename format: {canon_id}_alt_{rogue_id}.json or {canon_id}_rogue_{rogue_id}.json
    canon_id = int(dup_file.stem.split('_')[0])
    canon_file = cache / f"{canon_id}.json"
    
    if not canon_file.exists():
        continue
    
    try:
        main = json.loads(canon_file.read_text())
        alt  = json.loads(dup_file.read_text())
    except Exception:
        continue

    main_conf = conf_rank.get(main.get('annotation_confidence', 1), 1)
    alt_conf  = conf_rank.get(alt.get('annotation_confidence', 1), 1)

    # Count differing fields
    diffs = []
    for f in craft_fields:
        mv = str(main.get(f, ''))
        av = str(alt.get(f, ''))
        if mv != av:
            diffs.append(f)

    rows.append({
        'canon_id':    canon_id,
        'title':       main.get('title', '?'),
        'dup_file':    dup_file.name,
        'main_conf':   main.get('annotation_confidence', '?'),
        'alt_conf':    alt.get('annotation_confidence', '?'),
        'diff_count':  len(diffs),
        'diffs':       ', '.join(diffs),
        'alt_better':  alt_conf > main_conf,
        'same_conf':   alt_conf == main_conf,
        '_main_file':  canon_file,
        '_alt_file':   dup_file,
        '_main_data':  main,
        '_alt_data':   alt,
    })

df_dups = pd.DataFrame(rows)

print(f"Duplicate pairs found:          {len(df_dups)}")
print(f"Alt is higher confidence:       {df_dups['alt_better'].sum()}")
print(f"Same confidence:                {df_dups['same_conf'].sum()}")
print(f"Main is higher confidence:      {(~df_dups['alt_better'] & ~df_dups['same_conf']).sum()}")
print(f"Pairs with 0 field differences: {(df_dups['diff_count']==0).sum()}")
print(f"Pairs with 1-3 differences:     {((df_dups['diff_count']>0) & (df_dups['diff_count']<=3)).sum()}")
print(f"Pairs with 4+ differences:      {(df_dups['diff_count']>=4).sum()}")
print()
print("Confidence breakdown:")
print(df_dups.groupby(['main_conf', 'alt_conf']).size().reset_index(name='count').to_string())

Duplicate pairs found:          411
Alt is higher confidence:       0
Same confidence:                215
Main is higher confidence:      196
Pairs with 0 field differences: 0
Pairs with 1-3 differences:     2
Pairs with 4+ differences:      409

Confidence breakdown:
   main_conf  alt_conf  count
0          2         1      5
1          2         2    122
2          3         2    191
3          3         3     93


In [27]:
df_dups[~df_dups['alt_better'] & ~df_dups['same_conf']]

,canon_id,title,dup_file,main_conf,alt_conf,diff_count,diffs,alt_better,same_conf,_main_file,_alt_file,_main_data,_alt_data
0,670,Oldboy,670_alt_6314.json,3,2,11,"narrative_time, pacing_signature, ending_valen...",False,False,data/cache/annotations/670.json,data/cache/annotations_duplicates/670_alt_6314...,"{'tmdb_id': 670, 'title': 'Oldboy', 'year': No...","{'tmdb_id': 6314, 'title': 'Oldboy', 'year': N..."
1,705996,Decision to Leave,705996_alt_786293.json,3,2,8,"narrative_time, point_of_view, reality_registe...",False,False,data/cache/annotations/705996.json,data/cache/annotations_duplicates/705996_alt_7...,"{'tmdb_id': 705996, 'title': 'Decision to Leav...","{'tmdb_id': 786293, 'title': 'Decision to Leav..."
2,25364,Ace in the Hole,25364_alt_147.json,3,2,10,"narrative_time, ending_valence, dialogue_densi...",False,False,data/cache/annotations/25364.json,data/cache/annotations_duplicates/25364_alt_14...,"{'tmdb_id': 25364, 'title': 'Ace in the Hole',...","{'tmdb_id': 147, 'title': 'Ace in the Hole', '..."
5,950082,Palthu Janwar,950082_alt_230145.json,3,2,9,"point_of_view, ending_valence, dialogue_densit...",False,False,data/cache/annotations/950082.json,data/cache/annotations_duplicates/950082_alt_2...,"{'tmdb_id': 950082, 'title': 'Palthu Janwar', ...","{'tmdb_id': 230145, 'title': 'Palthu Janwar', ..."
9,502422,Thunder Road,502422_alt_273469.json,3,2,11,"narrative_time, point_of_view, ending_valence,...",False,False,data/cache/annotations/502422.json,data/cache/annotations_duplicates/502422_alt_2...,"{'tmdb_id': 502422, 'title': 'Thunder Road', '...","{'tmdb_id': 273469, 'title': 'Thunder Road', '..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
396,3082,Modern Times,3082_alt_42.json,3,2,12,"narrative_time, point_of_view, ending_valence,...",False,False,data/cache/annotations/3082.json,data/cache/annotations_duplicates/3082_alt_42....,"{'tmdb_id': 3082, 'title': 'Modern Times', 'ye...","{'tmdb_id': 42, 'title': 'Modern Times', 'year..."
404,466420,Killers of the Flower Moon,466420_alt_732846.json,3,2,12,"narrative_time, point_of_view, ending_valence,...",False,False,data/cache/annotations/466420.json,data/cache/annotations_duplicates/466420_alt_7...,"{'tmdb_id': 466420, 'title': 'Killers of the F...","{'tmdb_id': 732846, 'title': 'Killers of the F..."
406,1075794,Leo,1075794_alt_972183.json,3,2,13,"narrative_time, pacing_signature, ending_valen...",False,False,data/cache/annotations/1075794.json,data/cache/annotations_duplicates/1075794_alt_...,"{'tmdb_id': 1075794, 'title': 'Leo', 'year': N...","{'tmdb_id': 972183, 'title': 'Leo', 'year': No..."
407,10740,Birth,10740_alt_968.json,3,2,12,"narrative_time, point_of_view, reality_registe...",False,False,data/cache/annotations/10740.json,data/cache/annotations_duplicates/10740_alt_96...,"{'tmdb_id': 10740, 'title': 'Birth', 'year': N...","{'tmdb_id': 968, 'title': 'Birth', 'year': Non..."
